# 3. Practical Drifting: normalisation, temperature, batching, diagnostics

Notebook 1 gave a working drifting model on a 2-D toy. Everything between that and the paper's ImageNet result is
**engineering around the fact that $\mathbf{V}$ is a noisy Monte-Carlo estimate in a metric space you do not control**.

This notebook is the recipe, with each ingredient justified by an experiment:

1. feature normalisation — make $\tau$ dimension- and scale-free
2. drift normalisation — make $\eta$ meaningful
3. multiple temperatures
4. step size vs learning rate
5. batch structure and the sample queue
6. EMA, warmup, clipping
7. **diagnostics**: the four numbers you should always log
8. a catalogue of failure modes and how to recognise each
9. a reference training loop

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Feature normalisation: making $\tau$ scale-free

The kernel is $\exp(-\|\mathbf{x}-\mathbf{y}\|/\tau)$. If you change the units of your data, or move from a 4-dim
feature to a 512-dim one, the typical distance changes and your carefully tuned $\tau$ becomes meaningless. In high
dimensions this is severe: for roughly isotropic features, $\|\mathbf{x}-\mathbf{y}\|$ grows like $\sqrt{C}$.

The paper's fix (Eqs 18–22): rescale each feature by a scalar $S_j$ so that the *average* pairwise distance is
$\sqrt{C_j}$,
$$S_j = \frac{1}{\sqrt{C_j}}\,\mathbb{E}_{\mathbf{x}}\mathbb{E}_{\mathbf{y}}\big[\|\phi_j(\mathbf{x})-\phi_j(\mathbf{y})\|\big],
\qquad \tilde\phi_j = \phi_j / S_j$$
and then use $\tilde\tau_j = \tau\sqrt{C_j}$ in the kernel. After this, $\tau$ is a pure number: $\tau=0.05$ means
"nearby = 5% of the typical inter-sample distance", regardless of dimension or scale. $S_j$ is computed from the
current batch with a stop-gradient (conceptually it is a statistic of the *previous* batch).

In [ ]:
def feature_scale(f, fp):
    """S_j from paper eq. 21, using the batch's own pairwise distances."""
    C = f.shape[-1]
    with torch.no_grad():
        return (torch.cdist(f, fp).mean() / math.sqrt(C)).clamp_min(1e-8)

# demonstrate: raw distances explode with dimension, normalised ones do not
print(f"{'dim':>6s} {'raw mean dist':>15s} {'after /S':>10s}")
for C in [2, 16, 128, 1024]:
    a, b = torch.randn(256, C), torch.randn(256, C)
    S = feature_scale(a, b)
    print(f"{C:6d} {torch.cdist(a,b).mean():15.2f} {torch.cdist(a/S,b/S).mean()/math.sqrt(C):10.3f}")

Column 3 is $\approx 1$ in every dimension: after normalisation, distances measured in units of $\sqrt{C}$ are
$O(1)$. That is exactly the invariance we want.

## 2. Drift normalisation: making $\eta$ meaningful

The same problem afflicts the drift itself. $\mathbf{V}_j$ lives in feature space $j$; its magnitude depends on that
space. When you sum many losses (one per feature, per scale, per temperature) the largest-magnitude term dominates.
The fix (Eqs 23–25): rescale each drift to unit RMS per dimension,
$$\lambda_j=\sqrt{\mathbb{E}\big[\tfrac{1}{C_j}\|\mathbf{V}_j\|^2\big]},\qquad \tilde{\mathbf{V}}_j=\mathbf{V}_j/\lambda_j$$

**Consequence you must not forget** (established in Notebook 1 §7): the training loss is then pinned at
$\approx\eta^2\cdot\text{const}$ and stops being a convergence diagnostic. Log the *un-normalised* $\|\mathbf{V}\|^2$
separately.

In [ ]:
def normalised_drift(V):
    lam = V.pow(2).mean().sqrt().clamp_min(1e-8)      # E[||V||^2 / C]^{1/2}
    return V / lam

## 3–4. A reference implementation

Putting feature normalisation, drift normalisation and multi-temperature together, with the diagnostics we will need
in §7.

In [ ]:
def compute_V(x, y_pos, y_neg=None, tau=0.2):
    N = x.shape[0]
    self_neg = y_neg is None
    if self_neg:
        y_neg = x
    lp = -torch.cdist(x, y_pos) / tau
    ln = -torch.cdist(x, y_neg) / tau
    if self_neg:
        ln = ln - torch.eye(N, device=x.device) * 1e6
    logit = torch.cat([lp, ln], dim=1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    P = y_pos.shape[0]
    A_pos, A_neg = A[:, :P], A[:, P:]
    V = (A_pos * A_neg.sum(1, keepdim=True)) @ y_pos - (A_neg * A_pos.sum(1, keepdim=True)) @ y_neg
    return V

def drifting_loss(f, f_pos, taus=(0.02, 0.05, 0.2), eta=0.05, diagnostics=None):
    """Feature-normalised, drift-normalised, multi-temperature drifting loss.
    f      : [N, C] generated features (requires grad)
    f_pos  : [P, C] positive features
    """
    C = f.shape[-1]
    S = feature_scale(f, f_pos)
    fn, fpn = f / S, f_pos / S
    with torch.no_grad():
        V_total = 0.0
        for t in taus:
            V = compute_V(fn, fpn, None, tau=t * math.sqrt(C))
            if diagnostics is not None:
                diagnostics.setdefault(f"rawV_tau{t}", []).append(V.pow(2).mean().item())
            V_total = V_total + normalised_drift(V)
        target = fn + eta * V_total
    return F.mse_loss(fn, target)

Note the two-knob structure for transport speed:

* **$\eta$** — how far the frozen target sits from the current sample, measured in normalised drift units.
* **learning rate** — what fraction of that gap the optimiser actually closes in one step (Notebook 0c).

They multiply. Too much of either and the model chases a target it can never track; too little and training stalls.
Let's map the region that works.

In [ ]:
def spiral(n, device=device):
    t = torch.sqrt(torch.rand(n, device=device)) * 3.5 * math.pi
    r = 0.35 * t
    return (torch.stack([r * torch.cos(t), r * torch.sin(t)], -1)
            + 0.12 * torch.randn(n, 2, device=device)) / 3.9

class Gen(nn.Module):
    def __init__(self, d_z=2, d_out=2, h=256, nl=4):
        super().__init__()
        L, d = [], d_z
        for _ in range(nl):
            L += [nn.Linear(d, h), nn.SiLU()]; d = h
        L += [nn.Linear(d, d_out)]
        self.net = nn.Sequential(*L); self.d_z = d_z
    def forward(self, z): return self.net(z)

def tv2d(A, B, bins=24, lim=1.3):
    def h(z):
        H = torch.histogramdd(z.cpu(), bins=[bins, bins], range=[-lim, lim, -lim, lim])[0]
        return H / H.sum()
    return 0.5 * (h(A) - h(B)).abs().sum().item()

def train(eta=0.05, lr=2e-3, taus=(0.05, 0.2, 0.8), steps=2500, N=256, P=256,
          ema_decay=None, ema_warmup=True, warmup=0, clip=2.0, seed=0, collect=False):
    torch.manual_seed(seed)
    g = Gen().to(device)
    opt = torch.optim.Adam(g.parameters(), lr, betas=(0.9, 0.95), weight_decay=0.0)
    ema = {k: v.detach().clone() for k, v in g.state_dict().items()} if ema_decay else None
    diags = {} if collect else None
    for s in range(steps):
        if warmup and s < warmup:
            for gp in opt.param_groups:
                gp["lr"] = lr * (s + 1) / warmup
        x = g(torch.randn(N, 2, device=device))
        loss = drifting_loss(x, spiral(P), taus=taus, eta=eta, diagnostics=diags)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(g.parameters(), clip)
        opt.step()
        if ema is not None:
            # warm up the decay: early on, track the weights closely, so the average
            # is not dominated by the (bad) initial parameters. Without this, an EMA
            # whose horizon 1/(1-decay) is comparable to the run length *hurts*.
            d = min(ema_decay, (1 + s) / (10 + s)) if ema_warmup else ema_decay
            with torch.no_grad():
                for k, v in g.state_dict().items():
                    ema[k].mul_(d).add_(v.detach(), alpha=1 - d)
    if ema is not None:
        g.load_state_dict(ema)
    with torch.no_grad():
        out = g(torch.randn(20000, 2, device=device))
    return (out, diags) if collect else out

ref = spiral(20000)
print("TV floor:", round(tv2d(spiral(20000), ref), 3))
etas = [0.005, 0.02, 0.05, 0.2, 0.5]
lrs  = [2e-4, 1e-3, 5e-3]
tab = np.zeros((len(lrs), len(etas)))
for i, lr in enumerate(lrs):
    for j, eta in enumerate(etas):
        tab[i, j] = tv2d(train(eta=eta, lr=lr), ref)

fig, ax = plt.subplots(figsize=(5.6, 2.8))
im = ax.imshow(tab, cmap="viridis_r", vmin=0.04)
ax.set_xticks(range(len(etas)), [str(e) for e in etas]); ax.set_xlabel("drift scale eta")
ax.set_yticks(range(len(lrs)), [str(l) for l in lrs]); ax.set_ylabel("learning rate")
for i in range(len(lrs)):
    for j in range(len(etas)):
        ax.text(j, i, f"{tab[i,j]:.3f}", ha="center", va="center",
                color="w" if tab[i, j] > tab.mean() else "k", fontsize=8)
plt.colorbar(im, label="TV distance"); plt.title("the two transport knobs multiply"); plt.tight_layout(); plt.show()

A diagonal valley: what matters is roughly the product $\eta\times\text{lr}$, and the good region is a band, not a
point. This is why the paper can use one $\eta$-equivalent across configurations and only retune the learning rate
between model sizes.

## 5. Batch structure and the sample queue

For class-conditional generation the paper's batch is built as

* $N_c$ class labels (128 in the final runs)
* for **each** label: $N_{\text{neg}}=64$ generated samples, $N_{\text{pos}}$ real samples of that class,
  $N_{\text{unc}}=32$ real samples from random classes (for CFG, Notebook 5)
* effective batch $B = N_c\times N_{\text{neg}} = 8192$

The drift is computed **independently per label** — a sample is only repelled by other samples of the same class, and
only attracted to positives of the same class. One epoch is defined by the number of *generated* samples: $N_{\text{data}}/B$ steps.

Getting $N_{\text{pos}}$ fresh same-class real samples every step is a data-loading problem, not a modelling one. The
paper solves it with a **MoCo-style queue**: one FIFO of 128 samples per class, pushed 64 new samples per step; plus
one global queue of 1000 for unconditional negatives. Sampling is without replacement from the relevant queue.

In [ ]:
class SampleQueue:
    """Per-class FIFO of real samples, MoCo style (paper A.8)."""
    def __init__(self, n_classes, dim, size=128, device="cpu"):
        self.buf = torch.zeros(n_classes, size, dim, device=device)
        self.n = torch.zeros(n_classes, dtype=torch.long, device=device)
        self.ptr = torch.zeros(n_classes, dtype=torch.long, device=device)
        self.size = size
    def push(self, x, labels):
        for c in labels.unique():
            v = x[labels == c]
            for row in v:
                self.buf[c, self.ptr[c]] = row
                self.ptr[c] = (self.ptr[c] + 1) % self.size
                self.n[c] = min(self.n[c] + 1, self.size)
    def sample(self, c, k):
        avail = int(self.n[c])
        idx = torch.randperm(avail, device=self.buf.device)[:min(k, avail)]   # without replacement
        return self.buf[c, idx]

q = SampleQueue(3, 2, size=8)
for _ in range(4):
    q.push(torch.randn(6, 2), torch.randint(0, 3, (6,)))
print("queue occupancy per class:", q.n.tolist())
print("draw 4 from class 0:", q.sample(0, 4).shape)

Two properties worth noting. The queue makes positives **slightly stale** (up to 128/64 = 2 steps old for a busy
class), which is harmless because the data distribution is static. And it decouples the drift batch size from the
data-loader batch size — you can compute a field over 128 positives per class while only loading 64 new samples per
step.

## 6. EMA, warmup, gradient clipping

The drifting target is a Monte-Carlo estimate, so the parameter trajectory is noisy. The paper's settings
(Table 8) are:

| setting | ablation default | final (L/2 latent) |
|---|---|---|
| optimiser | AdamW $\beta=(0.9,0.95)$ | same |
| lr | 2e-4 | 4e-4 |
| weight decay | 0.01 | 0.01 |
| warmup | 5k steps | 10k steps |
| grad clip | 2.0 | 2.0 |
| EMA decay | 0.999 | $\{0.999 \ldots 0.9999\}$ |
| steps / epochs | 30k / 100 | 200k / 1280 |
| batch $B$ | 4096 | 8192 |

In [ ]:
print(f"{'config':34s} {'TV':>7s}")
for name, kw in [("plain",                          dict()),
                 ("+ EMA 0.99  (horizon 100)",       dict(ema_decay=0.99)),
                 ("+ EMA 0.999 (horizon 1000)",      dict(ema_decay=0.999)),
                 ("+ EMA 0.9995 (horizon 2000)",     dict(ema_decay=0.9995)),
                 ("+ EMA 0.9995, no decay warm-up",  dict(ema_decay=0.9995, ema_warmup=False)),
                 ("+ lr warmup 300",                 dict(warmup=300)),
                 ("+ EMA 0.99 + lr warmup",          dict(ema_decay=0.99, warmup=300)),
                 ("no grad clip",                    dict(clip=1e9))]:
    print(f"{name:34s} {tv2d(train(**kw), ref):7.3f}")
print(f"\n(this run is only 2500 steps; the paper's are 200k)")

**EMA is the single biggest win here** — it more than halves the TV distance (0.24 to about 0.10). That is not surprising once you remember
what the target is: a Monte-Carlo estimate of a field, so the parameter trajectory has a large stochastic component
that averaging removes. This is the same reason EMA is standard in the whole self-distillation family (Notebook 0c §5).

Two details that matter more than the decay value:

1. **Warm up the decay.** An EMA with decay $d$ has an effective horizon of $1/(1-d)$ steps: 100 at $d{=}0.99$ but
   **2000** at $d{=}0.9995$. In a 2500-step run, a naive $d{=}0.9995$ average is dominated by the useless initial
   weights and actively *hurts* — the no-warm-up row measures TV 0.82, against 0.24 with no EMA at all. The one-line
   fix in the code above, $d_s=\min(d,(1+s)/(10+s))$, tracks the weights closely early and relaxes to $d$ later; with
   it, every horizon helps.
2. **Match the horizon to the run length, not to a number copied from a paper.** The paper's 200k-step runs use
   $d\in\{0.999\ldots0.9999\}$, i.e. horizons of 1k–10k, a few percent of training. Scaled to a 2500-step toy that is
   $d\approx0.99$ — which ties for best above (0.102 against 0.103 for the longer horizons, once warmed up). The same caution applies to learning-rate warmup: 10k steps
   out of 200k is 5%; 10k out of 2500 is nonsense.

## 7. The four numbers you should always log

**(a) raw $\|\mathbf{V}_\tau\|^2$, per temperature.** Distance from equilibrium at each scale. Should decrease.
If the small-$\tau$ term is flat and tiny from step 0, the kernel is starved at that scale.

**(b) Effective sample size of the kernel weights,** $\mathrm{ESS}=1/\sum_j \bar k_j^2$, averaged over queries.
This is *the* diagnostic for temperature health. $\mathrm{ESS}\approx1$ means the field is "copy your nearest
neighbour" (memorisation risk, high variance). $\mathrm{ESS}\approx P$ means "move to the global mean" (moment
matching only). You want something in between, tens out of hundreds.

**(c) Attraction/repulsion balance,** $\|\mathbf{V}^{+}\|/\|\mathbf{V}^{-}\|$. Should approach 1 as you converge.
Persistently $>1$ means the model is being over-pulled and risks collapse (Notebook 1 §9).

**(d) Sample spread**, e.g. per-dimension std vs. the data's. Cheap early-warning for collapse or over-dispersion.

In [ ]:
def diagnostics(x, y_pos, taus=(0.05, 0.2, 0.8)):
    C = x.shape[-1]; S = feature_scale(x, y_pos)
    xn, ypn = x / S, y_pos / S
    rows = []
    for t in taus:
        tt = t * math.sqrt(C)
        V = compute_V(xn, ypn, None, tau=tt)
        wp = (-torch.cdist(xn, ypn) / tt).softmax(-1)          # kernel weights over the P positives
        ess = (1.0 / wp.pow(2).sum(-1)).mean().item()
        Vp = wp @ ypn - xn
        dn = -torch.cdist(xn, xn) / tt - torch.eye(xn.shape[0], device=x.device) * 1e6
        Vn = dn.softmax(-1) @ xn - xn
        rows.append((t, V.pow(2).mean().item(), ess,
                     Vp.norm(dim=-1).mean().item() / max(Vn.norm(dim=-1).mean().item(), 1e-9)))
    return rows

torch.manual_seed(0)
g = Gen().to(device); opt = torch.optim.Adam(g.parameters(), 2e-3)
log = {"step": [], "rawV": [], "ess": [], "bal": [], "std": []}
for s in range(2501):
    x = g(torch.randn(256, 2, device=device))
    loss = drifting_loss(x, spiral(256), taus=(0.05, 0.2, 0.8), eta=0.05)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(g.parameters(), 2.0); opt.step()
    if s % 50 == 0:
        with torch.no_grad():
            xd = g(torch.randn(512, 2, device=device))
            rows = diagnostics(xd, spiral(512))
        log["step"].append(s)
        log["rawV"].append([r[1] for r in rows])
        log["ess"].append([r[2] for r in rows])
        log["bal"].append([r[3] for r in rows])
        log["std"].append(xd.std().item())

taus = (0.05, 0.2, 0.8)
fig, ax = plt.subplots(1, 4, figsize=(15, 3.0))
A = np.array(log["rawV"]); B = np.array(log["ess"]); Cc = np.array(log["bal"])
for k, t in enumerate(taus):
    ax[0].semilogy(log["step"], A[:, k], label=f"tau={t}")
    ax[1].plot(log["step"], B[:, k], label=f"tau={t}")
    ax[2].plot(log["step"], Cc[:, k], label=f"tau={t}")
ax[0].set_title("(a) raw ||V||^2"); ax[1].set_title("(b) kernel ESS (of 512)")
ax[2].set_title("(c) |V+| / |V-|"); ax[2].axhline(1.0, ls="--", c="k")
ax[3].plot(log["step"], log["std"]); ax[3].axhline(spiral(20000).std().item(), ls="--", c="k")
ax[3].set_title("(d) sample std vs data")
for a in ax[:3]:
    a.legend(fontsize=7); a.grid(alpha=0.3); a.set_xlabel("step")
ax[3].grid(alpha=0.3); ax[3].set_xlabel("step")
plt.tight_layout(); plt.show()

Read the panels together. (a) raw $\|\mathbf{V}\|^2$ does *not* fall steadily here. It starts tiny at the two smaller
temperatures while the freshly initialised generator is still nearly a point (spread 0.06), then rises within a few hundred steps and plateaus noisily around $10^{-4}$–$10^{-3}$: on a 2-D toy the estimator's
noise floor arrives early, and the plateau is the signal that a scale has stopped improving. (b) ESS settles to a
sensible fraction of the batch (about 17, 84 and 377 of 512); the small-$\tau$ curve is the one to watch. (c) the
attraction/repulsion ratio converges to about 1 at the two larger temperatures, which is what equilibrium *means*; at
$\tau=0.05$ it stays near 1.3–1.5, the scale at which this short run is least converged. (d) the spread locks onto
the data's (0.49 against 0.48).

## 8. Failure-mode catalogue

| symptom | diagnosis | fix |
|---|---|---|
| raw $\|\mathbf{V}\|^2$ tiny and flat from step 0 | **starved kernel** — everything is in the exponential tail | raise $\tau$; add a feature encoder (NB 4); normalise features |
| sample std collapses to ~0 | **over-attraction** — anti-symmetry broken, or self-exclusion missing | check $\mathbf{V}^{+}/\mathbf{V}^{-}$ balance = 1; mask the diagonal |
| sample std much larger than data | **over-repulsion** | check for double-counted negatives (e.g. CFG negatives weighted wrongly) |
| ESS $\approx 1$ at every $\tau$ | field is nearest-neighbour copying | raise $\tau$; more positives; check the metric is meaningful |
| ESS $\approx P$ at every $\tau$ | moment matching only, distribution won't be identified | lower $\tau$; add small $\tau$ to the set |
| loss flat but samples improve | you normalised the drift — expected | log raw $\|\mathbf{V}\|^2$ |
| training diverges after some steps | target chasing: $\eta\times$lr too large | lower either; check grad clip is on |
| modes missing but samples sharp | positives per class too few, or per-class batches too small | raise $N_{\text{pos}}$; more classes per step |

## 9. Reference training loop

Everything above, in the shape you would actually write it.

In [ ]:
def train_reference(data_fn, d_z=2, d_out=2, steps=3000, N_neg=256, N_pos=256,
                    taus=(0.05, 0.2, 0.8), eta=0.05, lr=2e-3, wd=0.01,
                    warmup=200, clip=2.0, ema_decay=0.999, log_every=500, seed=0):
    torch.manual_seed(seed)
    g = Gen(d_z=d_z, d_out=d_out).to(device)
    opt = torch.optim.AdamW(g.parameters(), lr, betas=(0.9, 0.95), weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    ema = {k: v.detach().clone() for k, v in g.state_dict().items()}

    for s in range(steps):
        if s < warmup:
            for gp in opt.param_groups:
                gp["lr"] = lr * (s + 1) / warmup

        x = g(torch.randn(N_neg, d_z, device=device))         # 1 forward pass
        y_pos = data_fn(N_pos)

        C = x.shape[-1]
        S = feature_scale(x, y_pos)
        xn, ypn = x / S, y_pos / S
        with torch.no_grad():
            raws, V_tot = [], 0.0
            for t in taus:
                V = compute_V(xn, ypn, None, tau=t * math.sqrt(C))
                raws.append(V.pow(2).mean().item())
                V_tot = V_tot + normalised_drift(V)
            target = xn + eta * V_tot
        loss = F.mse_loss(xn, target)

        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(g.parameters(), clip)
        opt.step()
        if s >= warmup:
            sched.step()
        d = min(ema_decay, (1 + s) / (10 + s))
        with torch.no_grad():
            for k, v in g.state_dict().items():
                ema[k].mul_(d).add_(v.detach(), alpha=1 - d)

        if s % log_every == 0:
            print(f"step {s:5d}  loss {loss.item():.3e}  rawV " +
                  " ".join(f"{r:.2e}" for r in raws))

    g_ema = Gen(d_z=d_z, d_out=d_out).to(device); g_ema.load_state_dict(ema)
    return g, g_ema

g_raw, g_ema = train_reference(spiral)
with torch.no_grad():
    s_raw = g_raw(torch.randn(20000, 2, device=device))
    s_ema = g_ema(torch.randn(20000, 2, device=device))
print(f"\nTV raw weights {tv2d(s_raw, ref):.3f}   TV EMA weights {tv2d(s_ema, ref):.3f}   floor {tv2d(spiral(20000), ref):.3f}")

## Summary

* **Normalise features** so $\tau$ is dimensionless; **normalise the drift** so $\eta$ is meaningful — then neither
  needs retuning across feature spaces or model sizes.
* **Sum over several $\tau$.** No tuning, and never catastrophic — robustness, not a guaranteed win over the
  best single value (Notebook 2 §2).
* Transport speed $\approx \eta \times$ learning rate. Wide valley, so it is forgiving — but they multiply.
* Batches are structured as (classes) $\times$ (samples per class); positives come from a per-class queue.
* Log four things: raw $\|\mathbf{V}_\tau\|^2$, kernel ESS, $\|\mathbf{V}^{+}\|/\|\mathbf{V}^{-}\|$, sample spread.
  Those four identify every failure mode in §8.

Next: [4. Feature-Space Drifting](<4. Feature-Space Drifting.ipynb>) — the piece that makes drifting work in high dimensions.